# Common Skin Disease Classification — EfficientNet-B2 GPU Training
### Google Colab Ready-to-Run Training Notebook

**Project**: AI-Based Skin Disease Detection with Treatment Cost Estimation  
**Model Architecture**: EfficientNet-B2 (`efficientnet_b2` via `timm`)  
**Input Resolution**: 260 × 260 with Aspect-Ratio-Preserving Padding  
**Total Parameters**: ~7.7M (over 11× smaller than ViT-B/16's 85.8M parameters)  
**Training Strategy**: Staged gradual unfreezing, discriminative learning rates, Weighted Cross-Entropy Loss, AdamW, mixed precision.

**Target Categories (9 Classes)**:
1. Acne (`acne`)
2. Allergic Contact Dermatitis (`allergic_contact_dermatitis`)
3. Eczema (`eczema`)
4. Impetigo (`impetigo`)
5. Psoriasis (`psoriasis`)
6. Ringworm / Tinea Corporis (`ringworm`)
7. Scabies (`scabies`)
8. Seborrheic Dermatitis (`seborrheic_dermatitis`)
9. Urticaria / Hives (`urticaria`)

## 1. GPU Verification

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:     ', torch.cuda.get_device_name(0))
    print('VRAM:           ', f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print('WARNING: Running on CPU. Switch Colab runtime to GPU (Runtime -> Change runtime type -> T4 GPU).')

## 2. Environment Dependencies Installation

In [ ]:
!pip install -q timm torchvision scikit-learn pandas matplotlib Pillow

## 3. Extract Colab Package Archive
Upload `colab_common_efficientnet_package.zip` using the left-hand files pane, then execute the cell below.

In [ ]:
import os
from pathlib import Path

pkg_name = 'colab_common_efficientnet_package.zip'
if os.path.exists(pkg_name):
    !unzip -q {pkg_name}
    print(f'Extracted {pkg_name} successfully.')
elif os.path.exists('colab_common_skin_package.zip'):
    !unzip -q colab_common_skin_package.zip
    print('Extracted colab_common_skin_package.zip successfully.')
else:
    print('Package zip not found. Please upload colab_common_efficientnet_package.zip to Colab files pane.')

# Ensure output directories exist
os.makedirs('models', exist_ok=True)
os.makedirs('training/results/common_efficientnet_plots', exist_ok=True)

# Verify directory structure
img_count = len(list(Path('data/common_skin/images').glob('*'))) if Path('data/common_skin/images').exists() else 0
print(f'Images count: {img_count}')
manifests = [f.name for f in Path('data/manifests').glob('common_skin*.csv')] if Path('data/manifests').exists() else []
print('Manifests found:', manifests)

## 4. Verify Dataset Splits & Class Weights

In [ ]:
import json
import pandas as pd

train_df = pd.read_csv('data/manifests/common_skin_train.csv')
val_df = pd.read_csv('data/manifests/common_skin_val.csv')
test_df = pd.read_csv('data/manifests/common_skin_test.csv')

print(f'Train Samples:      {len(train_df):,} | Cases: {train_df["case_id"].nunique():,}')
print(f'Validation Samples: {len(val_df):,} | Cases: {val_df["case_id"].nunique():,}')
print(f'Test Samples:       {len(test_df):,} | Cases: {test_df["case_id"].nunique():,}')
print(f'Total:              {len(train_df) + len(val_df) + len(test_df):,} clinical images')

with open('data/manifests/common_skin_metadata.json') as f:
    metadata = json.load(f)

print('\nClass weights:', metadata.get('class_weights'))

## 5. Verify EfficientNet-B2 Model Architecture

In [ ]:
from models.common_skin_efficientnet import CommonSkinEfficientNetB2

model = CommonSkinEfficientNetB2(num_classes=9, pretrained=True)
counts = model.count_parameters()
print(f'Total Parameters:     {counts["total_parameters"]:,}')
print(f'Trainable Parameters: {counts["trainable_parameters"]:,}')
print(f'Frozen Parameters:    {counts["frozen_parameters"]:,}')

## 6. Run EfficientNet-B2 Training Pipeline (GPU)

In [ ]:
from training.train_common_efficientnet import train_common_efficientnet

results = train_common_efficientnet(
    epochs=25,
    batch_size=16,
    lr=3e-4,
    patience=5,
    smoke_test=False,
    device_str='cuda' if torch.cuda.is_available() else 'cpu'
)

## 7. Display Training Curves & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Display loss & F1 curves
curves_img_path = 'training/results/common_efficientnet_plots/common_skin_efficientnet_training_curves.png'
if os.path.exists(curves_img_path):
    img = mpimg.imread(curves_img_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training & Validation Curves', fontsize=14, fontweight='bold')
    plt.show()

# Display confusion matrix
cm_img_path = 'training/results/common_efficientnet_plots/common_skin_efficientnet_confusion_matrix.png'
if os.path.exists(cm_img_path):
    img = mpimg.imread(cm_img_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
    plt.show()

## 8. Summary of Held-Out Test Evaluation

In [ ]:
with open('training/results/common_skin_efficientnet_metrics.json') as f:
    metrics = json.load(f)

print('===================================================')
print('EFFICIENTNET-B2 TEST EVALUATION SUMMARY')
print('===================================================')
print('Overall Accuracy:     ', f"{metrics['overall_accuracy']*100:.2f}%")
print('Balanced Accuracy:    ', f"{metrics['balanced_accuracy']*100:.2f}%")
print('Macro Precision:      ', f"{metrics['macro_precision']*100:.2f}%")
print('Macro Recall:         ', f"{metrics['macro_recall']*100:.2f}%")
print('Macro F1-Score:       ', f"{metrics['macro_f1']*100:.2f}%")
print('Weighted F1-Score:    ', f"{metrics['weighted_f1']*100:.2f}%")
print('Multiclass ROC-AUC:   ', metrics.get('roc_auc_macro'))
print('Expected Calib Error: ', f"{metrics.get('expected_calibration_error', 0)*100:.2f}%")
print('CPU Latency (mean):   ', f"{metrics.get('cpu_latency', {}).get('mean_latency_ms', 0):.2f} ms")

print('\n--- Per-Class Performance ---')
per_class_df = pd.DataFrame(metrics['per_class_metrics']).T
print(per_class_df.to_string())

## 9. Package & Download Artifacts
Download the trained checkpoint and results archive to your local computer.

In [ ]:
import zipfile
from google.colab import files

# Create results archive
zip_filename = 'common_skin_efficientnet_results.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk('training/results'):
        for fn in fnames:
            if 'efficientnet' in fn or 'efficientnet' in root:
                fpath = os.path.join(root, fn)
                z.write(fpath, os.path.relpath(fpath, 'training/results'))

print('Downloading models/common_skin_efficientnet.pth...')
files.download('models/common_skin_efficientnet.pth')

print('Downloading common_skin_efficientnet_results.zip...')
files.download(zip_filename)